In [1]:
import pandas as pd
import numpy as np
import json
import requests

In [ ]:
df = pd.read_parquet("../data/trips_pointv3_cleaned.parquet")

# try with a single trip
df = df[df.unique_id=='24c83c4e-1e60-4b65-9100-8cf1550076df_2022-01-07']
# sort pts by order in time
df = df.sort_values(by=['point_timestamp'])


In [4]:
df = pd.DataFrame({'longitude':df.point_longitude, 'latitude': df.point_latitude, 'gps_time': df.point_timestamp})
df = df[['gps_time', 'longitude', 'latitude']]
df.head()

df['gps_time'] = pd.to_datetime(df['gps_time'])
df = df.sort_values(by=['gps_time'])

df_trip_for_meili = df[['longitude', 'latitude', 'gps_time']].copy()
df_trip_for_meili.columns = ['lat', 'lon', 'time']
df_trip_for_meili.head()

,lat,lon,time
1449,11.118389,46.054911,2022-01-07 20:44:49+00:00
1450,11.118389,46.054911,2022-01-07 20:45:01+00:00
1451,11.119268,46.054611,2022-01-07 20:45:17+00:00
1452,11.119625,46.054618,2022-01-07 20:45:27+00:00
1453,11.120129,46.054765,2022-01-07 20:45:38+00:00


In [8]:
# preparing request 
# https://github.com/valhalla/valhalla-docs/blob/master/turn-by-turn/api-reference.md
meili_coordinates = df_trip_for_meili.to_json(orient='records')
meili_head = '{"shape":'
meili_tail = ""","search_radius": 300, "shape_match":"map_snap", "costing":"motor_scooter", "format":"osrm"}"""
meili_request_body = meili_head + meili_coordinates + meili_tail

In [9]:
# send request

url = "http://localhost:8002/trace_route"
headers = {'Content-type': 'application/json'}
data = str(meili_request_body)

r = requests.post(url, data=data, headers=headers)

In [10]:
# parsing the response 

if r.status_code == 200:
    
        response_text = json.loads(r.text)

        resp = str(response_text['tracepoints'])

        resp = resp.replace("'waypoint_index': None", "'waypoint_index': '#'")
        resp = resp.replace("None", "{'matchings_index': '#', 'name': '', 'waypoint_index': '#', 'alternatives_count': 0, 'distance': 0, 'location': [0.0, 0.0]}")

        resp = resp.replace("'", '"')

        resp = json.dumps(resp)
        resp = json.loads(resp)
        
        df_response = pd.read_json(resp)
        df_response = df_response[['name', 'distance', 'location']]
                
        df_trip_optimized = pd.merge(df_trip_for_meili, df_response, left_index=True, right_index=True)
else:
    print(r.status_code,'\n',r.text)

400 
 {"code":"NoSegment","message":"One of the supplied input coordinates could not snap to street segment."}
